# Day 2 Football detection, tracking and teams

Under **Runtime**, select **Change runtime type** and choose **T4 GPU**. Run the 15-second baseline first. The second trial defaults to 60 seconds and enables the optional higher-resolution ball pass. Each trial writes its own video, tracks, and metrics to Google Drive.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

from google.colab import drive

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Sports Vision Tracking')
assert (ROOT / 'scripts/track_football.py').is_file(), f'Project files are missing under {ROOT}'
assert (ROOT / 'Data/Football.mp4').is_file(), 'Place Football.mp4 under Data/'
os.chdir(ROOT)
subprocess.run(['nvidia-smi', '-L'], check=True)
%pip install -q -r requirements.txt

In [ ]:
import torch

assert torch.cuda.is_available(), 'Switch this notebook to a GPU runtime and rerun setup'
print(torch.cuda.get_device_name(0))
OUTPUTS = ROOT / 'outputs/football'

def run_trial(name, seconds, start_frame, ball_fallback=False):
    output = OUTPUTS / name
    command = [
        sys.executable, 'scripts/track_football.py',
        '--device', 'cuda:0', '--start-frame', str(start_frame),
        '--seconds', str(seconds), '--output-dir', str(output),
    ]
    if ball_fallback:
        command.append('--ball-fallback')
    subprocess.run(command, check=True)
    metrics = json.loads((output / 'metrics.json').read_text())
    assert metrics['frames_processed'] == round(seconds * 25)
    for artifact in ('annotated.mp4', 'tracks.json', 'tracks.csv'):
        assert (output / artifact).stat().st_size > 0
    print(f'Artifacts saved to {output}')
    return metrics

## 15-second smoke trial

Open the resulting MP4 to inspect IDs, labels, and bounding boxes before running the longer trial.

In [ ]:
smoke_metrics = run_trial('smoke', seconds=15, start_frame=1500)

## Configurable one-minute trial

Use the same start and duration with the fallback disabled if you want to compare throughput and ball detection coverage. Inspect ball detections visually for false positives.

In [ ]:
START_FRAME = 1500
SECONDS = 60
BALL_FALLBACK = True

trial_metrics = run_trial('one_minute', SECONDS, START_FRAME, BALL_FALLBACK)